In [ ]:
# ================================
# CELL 1 — Install
# ================================
!pip install rank_bm25 -q
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("✅ Done!")

In [ ]:
# ================================
# CELL 2 — Load Data
# ================================
import pickle
from google.colab import files

print("Upload processed_data.pkl")
files.upload()

with open('processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

statute_texts   = data['statute_texts']
statute_ids     = data['statute_ids']
precedent_texts = data['precedent_texts']
precedent_ids   = data['precedent_ids']
query_texts     = data['query_texts']
query_ids       = data['query_ids']
gold_statutes   = data['gold_statutes']
gold_precedents = data['gold_precedents']

print(f"✅ Statutes     : {len(statute_texts)}")
print(f"✅ Precedents   : {len(precedent_texts)}")
print(f"✅ Queries      : {len(query_texts)}")

In [ ]:
# ================================
# CELL 3 — Evaluation Functions
# ================================
import numpy as np

def precision_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    hits = len(set(retrieved_k) & set(relevant))
    return hits / k if k > 0 else 0.0

def recall_at_k(retrieved, relevant, k):
    retrieved_k = retrieved[:k]
    hits = len(set(retrieved_k) & set(relevant))
    return hits / len(relevant) \
        if len(relevant) > 0 else 0.0

def f1_at_k(retrieved, relevant, k):
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)

def mrr(retrieved, relevant):
    for rank, doc_id in enumerate(
            retrieved, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def map_score(retrieved, relevant, k):
    hits = 0
    sum_precisions = 0.0
    for i, doc_id in enumerate(
            retrieved[:k], start=1):
        if doc_id in relevant:
            hits += 1
            sum_precisions += hits / i
    return sum_precisions / len(relevant) \
        if len(relevant) > 0 else 0.0

def evaluate(all_retrieved, all_relevant, k):
    f1_scores  = []
    map_scores = []
    mrr_scores = []
    for retrieved, relevant in zip(
            all_retrieved, all_relevant):
        if len(relevant) == 0:
            continue
        f1_scores.append(
            f1_at_k(retrieved, relevant, k))
        map_scores.append(
            map_score(retrieved, relevant, k))
        mrr_scores.append(
            mrr(retrieved, relevant))
    return {
        f'F1@{k}'  : np.mean(f1_scores)  * 100,
        f'MAP@{k}' : np.mean(map_scores) * 100,
        'MRR'      : np.mean(mrr_scores) * 100
    }

print("✅ Evaluation functions ready!")

In [ ]:
# ================================
# CELL 4 — BM25 Functions
# ================================
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize

def build_ngrams(tokens, n):
    if n == 1:
        return tokens
    return [' '.join(tokens[i:i+n])
            for i in range(len(tokens)-n+1)]

def tokenize_ngram(text, n):
    if isinstance(text, list):
        text = ' '.join(text)
    tokens = word_tokenize(text.lower())
    return build_ngrams(tokens, n)

def run_bm25(corpus_texts, corpus_ids,
             query_texts, query_ids,
             gold_labels, ngram,
             task_name, k=10):

    print(f"\nBuilding BM25 {ngram}-gram "
          f"index for {task_name}...")
    tokenized_corpus = [
        tokenize_ngram(text, ngram)
        for text in corpus_texts]

    bm25 = BM25Okapi(
        tokenized_corpus, k1=1.6, b=0.75)

    print(f"Retrieving top-{k} for "
          f"{len(query_texts)} queries...")
    all_retrieved = []

    for i, query in enumerate(query_texts):
        if i % 100 == 0:
            print(f"  Query {i}/"
                  f"{len(query_texts)}...")
        tokenized_query = tokenize_ngram(
            query, ngram)
        scores = bm25.get_scores(tokenized_query)
        ranked_indices = sorted(
            range(len(scores)),
            key=lambda x: scores[x],
            reverse=True)[:k]
        retrieved_ids = [
            corpus_ids[idx]
            for idx in ranked_indices]
        all_retrieved.append(retrieved_ids)

    best_k       = k
    best_f1      = 0
    best_results = None

    for eval_k in range(1, k+1):
        results = evaluate(
            all_retrieved, gold_labels, eval_k)
        if results[f'F1@{eval_k}'] > best_f1:
            best_f1      = results[f'F1@{eval_k}']
            best_k       = eval_k
            best_results = results

    print(f"\n{'='*50}")
    print(f"BM25 {ngram}-gram | "
          f"{task_name} | Best k={best_k}")
    print(f"{'='*50}")
    for metric, value in best_results.items():
        print(f"  {metric:10s}: {value:.2f}%")

    return best_results, all_retrieved

print("✅ BM25 functions ready!")

In [ ]:
# ================================
# CELL 5 — Run BM25 LSR
# ================================
lsr_results   = {}
lsr_retrieved = {}

for ngram in [2, 3, 4, 5]:
    print(f"\n{'='*60}")
    print(f"BM25 {ngram}-gram — LSR")
    print(f"{'='*60}")
    results, retrieved = run_bm25(
        corpus_texts = statute_texts,
        corpus_ids   = statute_ids,
        query_texts  = query_texts,
        query_ids    = query_ids,
        gold_labels  = gold_statutes,
        ngram        = ngram,
        task_name    = "LSR",
        k            = 10)
    lsr_results[f'BM25_{ngram}gram']   = results
    lsr_retrieved[f'BM25_{ngram}gram'] = retrieved

print("\n✅ LSR BM25 COMPLETE!")

In [ ]:
# ================================
# CELL 6 — Run BM25 PCR
# ================================
pcr_results   = {}
pcr_retrieved = {}

for ngram in [2, 3, 4, 5]:
    print(f"\n{'='*60}")
    print(f"BM25 {ngram}-gram — PCR")
    print(f"{'='*60}")
    results, retrieved = run_bm25(
        corpus_texts = precedent_texts,
        corpus_ids   = precedent_ids,
        query_texts  = query_texts,
        query_ids    = query_ids,
        gold_labels  = gold_precedents,
        ngram        = ngram,
        task_name    = "PCR",
        k            = 10)
    pcr_results[f'BM25_{ngram}gram']   = results
    pcr_retrieved[f'BM25_{ngram}gram'] = retrieved

print("\n✅ PCR BM25 COMPLETE!")

In [ ]:
# ================================
# CELL 7 — Save & Download
# ================================
import pickle
from google.colab import files

with open('bm25_results.pkl', 'wb') as f:
    pickle.dump({
        'lsr_results'   : lsr_results,
        'lsr_retrieved' : lsr_retrieved,
        'pcr_results'   : pcr_results,
        'pcr_retrieved' : pcr_retrieved,
    }, f)

print("✅ Saved!")
files.download('bm25_results.pkl')
print("✅ Downloaded!")